# Regional consumption report 2023 — corrected version

Same goal as `mock_02_customer_data_merge.ipynb`. Each fix is marked with **Fix N** and refers to
the numbering in `mock_02_solution.md`.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## Load data

In [2]:
meters = pd.read_csv("../../data/meters.csv")
readings = pd.read_csv("../../data/meter_readings_daily.csv")
print(meters.shape, readings.shape)
print(meters["region"].unique())

(300, 7) (107503, 3)
['London' 'Scotland' 'Midlands' 'North' 'Wales' 'london' 'wales' 'north'
 'midlands']


**Fix 1.** The `unique()` output already showed `london`, `wales`, ... as separate values. Normalise
the key before any groupby, otherwise you get 9 regions instead of 5 and the small lowercase groups
distort every average.

In [3]:
meters["region"] = meters["region"].str.strip().str.title()
print(meters["region"].value_counts().to_dict())

{'London': 99, 'North': 66, 'Scotland': 57, 'Midlands': 45, 'Wales': 33}


**Fix 2.** Missing tariff is *unknown*, not Fixed. Keep it as NaN (or an explicit "Unknown" label) and
report it; 13 meters is 4% of the book.  
**Fix 3.** Missing estimate is *unknown*, not 0. Filling with 0 (and later adding 1 to dodge the
division) produced ratios in the thousands.

In [4]:
meters["tariff"] = meters["tariff"].fillna("Unknown")
meters["signup_date"] = pd.to_datetime(meters["signup_date"])
print("estimate missing:", meters["annual_kwh_estimate"].isna().sum())
meters["tariff"].value_counts()

estimate missing: 6


tariff
Fixed       143
Variable     94
TOU          50
Unknown      13
Name: count, dtype: int64

**Fix 4.** The 150 kWh/day "fault" threshold removed 257 readings, essentially all from SME meters
(SME mean is 70 kWh/day, residential max is 41). That silently trims the largest customers. Check who is
affected before deciding; here the readings are kept and flagged instead.

In [5]:
readings["date"] = pd.to_datetime(readings["date"])
big = readings["kwh"] > 150
who = readings.loc[big].merge(meters[["meter_id", "customer_type"]], on="meter_id")
print("readings > 150 kWh:", big.sum(), "| by customer type:", who["customer_type"].value_counts().to_dict())
readings["flag_high"] = big

readings > 150 kWh: 257 | by customer type: {'sme': 257}


**Fix 5.** Use `how="left"` with `indicator=True` and check the row count. The inner join silently
dropped 200 readings belonging to a meter that is not in the master (M999999). That is a data-quality
finding to report, not something to lose.

In [6]:
merged = readings.merge(meters, on="meter_id", how="left", indicator=True, validate="many_to_one")
print(merged["_merge"].value_counts().to_dict())
print("orphan meters:", merged.loc[merged["_merge"] == "left_only", "meter_id"].unique())
merged = merged[merged["_merge"] == "both"].drop(columns="_merge")
assert len(merged) == len(readings) - 200

{'both': 107303, 'left_only': 200, 'right_only': 0}
orphan meters: ['M999999']


**Fix 6.** The rates table has two rows for TOU (peak/offpeak). Merging on `tariff` alone duplicated
every TOU reading, so TOU meters' totals doubled (107,303 → 125,191 rows). Without an hourly split there is
no peak/offpeak information in daily data, so use a single blended TOU rate and validate the merge.

In [7]:
rates = pd.DataFrame({
    "tariff":      ["Fixed", "Variable", "TOU"],
    "unit_rate_p": [24.5,    27.1,       (35.0 + 12.0) / 2],
})
n_before = len(merged)
merged = merged.merge(rates, on="tariff", how="left", validate="many_to_one")
assert len(merged) == n_before
merged["cost_gbp"] = merged["kwh"] * merged["unit_rate_p"] / 100
print(merged.shape, "| rows without a rate (Unknown tariff):", merged["unit_rate_p"].isna().sum())

(107303, 12) | rows without a rate (Unknown tariff): 4641


**Fix 7.** `size()` counts rows including NaN; `count()` counts non-missing values. Report the one you
mean. **Fix 8.** Meters do not all have 365 readings (2% of days are missing), so compare *annualised*
consumption with the annual estimate, not the raw sum.

In [8]:
per_meter = (
    merged.groupby(["meter_id", "region", "tariff", "customer_type"], as_index=False)
          .agg(total_kwh=("kwh", "sum"),
               n_days=("kwh", "count"),
               total_cost=("cost_gbp", "sum"))
)
per_meter["annualised_kwh"] = per_meter["total_kwh"] * 365 / per_meter["n_days"]
per_meter = per_meter.merge(meters[["meter_id", "annual_kwh_estimate"]], on="meter_id")
per_meter["actual_vs_estimate"] = per_meter["annualised_kwh"] / per_meter["annual_kwh_estimate"]
print("coverage days: min", per_meter["n_days"].min(), "median", per_meter["n_days"].median())
per_meter[["annualised_kwh", "actual_vs_estimate"]].describe().round(2)

coverage days: min 350 median 358.0


,annualised_kwh,actual_vs_estimate
count,300.00,294.00
mean,5754.08,1.03
std,7433.53,0.01
min,975.72,0.99
25%,2795.13,1.02
50%,3408.15,1.03
75%,4126.80,1.04
max,38152.27,1.07


**Fix 9.** Regional averages are dominated by the SME share (an SME uses ~7x a household). Report the
median as well as the mean, and split by customer type before comparing regions.

In [9]:
summary = per_meter.groupby("region").agg(
    meters=("meter_id", "nunique"),
    sme_share=("customer_type", lambda s: (s == "sme").mean()),
    mean_kwh=("annualised_kwh", "mean"),
    median_kwh=("annualised_kwh", "median"),
    actual_vs_estimate_median=("actual_vs_estimate", "median"),
)
summary.round(2)

,meters,sme_share,mean_kwh,median_kwh,actual_vs_estimate_median
region,,,,,
London,99,0.12,6045.85,3528.20,1.03
Midlands,45,0.16,6490.31,3433.25,1.03
North,66,0.11,5694.96,3186.04,1.03
Scotland,57,0.05,4275.31,3187.68,1.03
Wales,33,0.15,6547.25,3414.88,1.03


In [10]:
by_type = per_meter.pivot_table(index="region", columns="customer_type",
                                values="annualised_kwh", aggfunc="median").round(0)
by_type

customer_type,residential,sme
region,,
London,3414.0,22907.0
Midlands,3314.0,22529.0
North,3129.0,27741.0
Scotland,3065.0,24236.0
Wales,3228.0,24309.0


**Fix 10.** `pivot_table` defaults to `aggfunc="mean"`; the heading said *total*. Say what you mean.

In [11]:
merged.pivot_table(index="region", columns="tariff", values="kwh", aggfunc="sum", margins=True).round(0)

tariff,Fixed,TOU,Unknown,Variable,All
region,,,,,
London,282334.0,109520.0,52175.0,142779.0,586808.0
Midlands,65769.0,73176.0,39268.0,108359.0,286571.0
North,171011.0,35681.0,2525.0,159188.0,368405.0
Scotland,117101.0,22705.0,4988.0,93561.0,238355.0
Wales,149200.0,17449.0,NaN,45583.0,212232.0
All,785415.0,258531.0,98956.0,549470.0,1692372.0


**Fix 11.** The commercial team's region labels are upper case, so the merge returned an *empty* frame.
The original printed `summary.head()` instead of `plan`, so nobody noticed. Normalise keys and assert.

In [12]:
region_targets = pd.DataFrame({
    "region": ["LONDON", "MIDLANDS", "NORTH", "SCOTLAND", "WALES"],
    "target_reduction_pct": [5, 3, 3, 2, 3],
})
region_targets["region"] = region_targets["region"].str.title()
plan = summary.reset_index().merge(region_targets, on="region", how="left", validate="one_to_one")
assert len(plan) == len(summary) and plan["target_reduction_pct"].notna().all()
plan[["region", "meters", "median_kwh", "target_reduction_pct"]]

,region,meters,median_kwh,target_reduction_pct
0,London,99,3528.200912,5
1,Midlands,45,3433.247792,3
2,North,66,3186.041897,3
3,Scotland,57,3187.676440,2
4,Wales,33,3414.881723,3


## Results (honest)

In [13]:
mean_ratio = summary.loc["London", "mean_kwh"] / summary.loc["Scotland", "mean_kwh"]
med_ratio = summary.loc["London", "median_kwh"] / summary.loc["Scotland", "median_kwh"]
res_ratio = by_type.loc["London", "residential"] / by_type.loc["Scotland", "residential"]
print(f"{per_meter['meter_id'].nunique()} meters matched; 200 readings belong to an unknown meter (M999999).")
print(f"London / Scotland: mean {mean_ratio:.2f}x, median {med_ratio:.2f}x, residential-only median {res_ratio:.2f}x.")
print(f"London has {summary.loc['London','sme_share']:.0%} SME meters vs {summary.loc['Scotland','sme_share']:.0%} in Scotland;")
print("the regional gap is customer mix, not household behaviour.")
print(f"Customers use {summary['actual_vs_estimate_median'].median():.2f}x their estimate: the estimates are fine.")

300 meters matched; 200 readings belong to an unknown meter (M999999).
London / Scotland: mean 1.41x, median 1.11x, residential-only median 1.11x.
London has 12% SME meters vs 5% in Scotland;
the regional gap is customer mix, not household behaviour.
Customers use 1.03x their estimate: the estimates are fine.
